In [ ]:
# ── 第一格：环境准备（D02 封装之后，每本 notebook 开头都是这十几行）──
%load_ext autoreload
%autoreload 2
# 改了 src/ 里的 .py，notebook 不用重启就能用上新代码。
from signal_mysql import database as mysql
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from signal_insight import config, geo, privacy, quality, viz
from signal_insight.viz import MAIN, MAIN_LIGHT, MAIN_DARK, MAIN_MID, GRAY, GRAY_DARK, RED, AMBER, INK, MUTED, FAINT, AREA_COLOR, SEG_COLOR    # 配色常量
viz.apply_theme()                           # 字体、配色、坐标轴样式：一行
con = mysql.connect(config.DB_SCHEMA)       # 路径从包里来：notebook 在哪个文件夹打开都找得到库
DAY = config.ANALYSIS_DAY                   # 分析日期（口径常量）
NB = "D09_专题分析"
SOURCE = f"{config.SOURCE_NOTE}；脚本：{NB}"    # 每张图左下角的来源行
pd.set_option("display.max_columns", 20)

In [ ]:
from signal_mysql import database as mysql
# 【跟丢了才运行】D6–D8 没做完的同学：把 False 改成 True 再运行，从检查点库复制标准表
CATCH_UP = False
if CATCH_UP:
    with mysql.connect(config.CHECKPOINT_SCHEMA) as ck:
        for t in ["dwd_signal", "dim_cell", "dim_hotspot", "dws_stay", "dws_trip", "dws_night", "dws_user_day"]:
            mysql.write_frame(mysql.read_frame(f"SELECT * FROM {t}", ck), t, con, if_exists="replace", index=False)
            print("已恢复", t)

In [ ]:
from signal_mysql import database as mysql
from signal_insight import db, dws, pipeline
settings = config.load_settings()
hot = mysql.read_frame("SELECT * FROM dim_hotspot", con).iloc[0]                  # 西塔中心（D4）
DAY_START = pd.Timestamp(DAY)                                                  # 2018-10-03 00:00
DISTRICT = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_district_boundary", con), "district")    # 区县边界（D3）
AOI = geo.rings_from_points(mysql.read_frame("SELECT * FROM ods_aoi_v2", con), "name")                        # 商圈边界，D3 选定的 v2

In [ ]:
from signal_mysql import database as mysql
stay = mysql.read_frame("SELECT * FROM dws_stay", con, parse_dates=["start_time", "end_time"])
day = mysql.read_frame("SELECT * FROM dws_user_day", con)
sig = mysql.read_frame("""SELECT s.imsi, s.ts, c.lng, c.lat FROM dwd_signal s
                     JOIN dim_cell c ON s.lac_cell = c.lac_cell""", con, parse_dates=["ts"])
print(f"用户 {day.shape[0]:,}；停留 {len(stay):,}；信令 {len(sig):,}")

In [ ]:
# 这一格在做什么：造一张"每人每小时一个位置"的表 dws_user_hour。
#
# 为什么需要它：前面的表要么是"一个人一行"（dws_user_day），要么是"一次停留一行"（dws_stay），
# 都没法回答"19 点有多少人在西塔"。要回答这类问题，就得把时间切成等长的格子，
# **每人每小时只保留一个位置**。
#
# 位置从哪来，分两种情况（优先级从高到低）：
#   ① 有停留覆盖这个小时 ≥ 30 分钟 → 用停留的质心（更稳，代表"他这一小时主要在这"）；
#   ② 没有 → 退回用这一小时所有信令位置的中位数。
# 下面的 cover 就是在算"这个停留覆盖了第 h 小时多少分钟"，算法和 D7 的重叠时长一样：
#   min(结束, 小时右端) − max(开始, 小时左端)，负数当 0。
#
# 讲台提示：跑完看一眼"来自停留 / 来自信令"的比例，这个比例本身就是数据质量的一个指标。
sh = (stay["start_time"] - DAY_START).dt.total_seconds() / 3600      # 停留的开始、结束换算成"当天第几小时"
eh = (stay["end_time"] - DAY_START).dt.total_seconds() / 3600
parts = []
for h in range(24):
    cover = (np.minimum(eh, h + 1) - np.maximum(sh, h)).clip(lower=0) * 60    # 这个停留覆盖了第 h 小时多少分钟
    m = cover >= 30
    parts.append(stay.loc[m, ["imsi", "lng", "lat"]].assign(hour=h, cover=cover[m]))
by_stay = (pd.concat(parts).sort_values("cover", ascending=False)
           .drop_duplicates(["imsi", "hour"]).drop(columns="cover").assign(src="停留"))
by_sig = (sig.assign(hour=sig["ts"].dt.hour)
          .groupby(["imsi", "hour"], as_index=False)[["lng", "lat"]].median())
# indicator=True 会多出一列 _merge，取值 both / left_only / right_only。
# 这里用它做"反连接"（antijoin）：只保留 left_only，也就是**没被停留覆盖到**的那些人 × 小时。
# 这是 pandas 里实现"A 减去 B"的标准写法，比先算集合再 isin 快得多。
by_sig = by_sig.merge(by_stay[["imsi", "hour"]], how="left", indicator=True)      # 已经有停留位置的"人 × 小时"不再用信令
by_sig = by_sig[by_sig["_merge"] == "left_only"].drop(columns="_merge").assign(src="信令")
uh = pd.concat([by_stay, by_sig], ignore_index=True)
uh["district"] = pd.Series(geo.point_in_polygons(uh["lng"], uh["lat"], DISTRICT)).fillna("市外").to_numpy()
uh["aoi"] = geo.point_in_polygons(uh["lng"], uh["lat"], AOI)
uh["dist_xita_m"] = geo.haversine(uh["lat"], uh["lng"], hot["lat"], hot["lng"])
uh = uh.sort_values(["imsi", "hour"], ignore_index=True)
print(f"{len(uh):,} 行（人 × 小时）：来自停留 {(uh['src'] == '停留').mean():.0%}，来自信令中位位置 {(uh['src'] == '信令').mean():.0%}")
uh.head()

In [ ]:
from signal_mysql import database as mysql
mysql.write_frame(uh, "dws_user_hour", con, if_exists="replace", index=False)
mysql.read_frame("SELECT hour, COUNT(*) AS 可定位人数 FROM dws_user_hour GROUP BY hour", con).set_index("hour").T

In [ ]:
# 对账
n_by_hour = uh.groupby("hour").size()
assert len(uh) == 102_859 and n_by_hour[19] == 5_069 and n_by_hour[4] == 3_666
print(f"✅ 对账通过：dws_user_hour {len(uh):,} 行；19 点可定位 {n_by_hour[19]:,} 人，凌晨 4 点 {n_by_hour[4]:,} 人")

In [ ]:
# 包里的版本和上面一步步算的 uh 逐行对账
user_hour = dws.build_user_hour(stay, sig, DAY, settings, DISTRICT, AOI, hot)
pd.testing.assert_frame_equal(user_hour, uh[user_hour.columns], check_dtype=False)
db.replace_table(con, user_hour, "dws_user_hour")
log = pipeline.run_day()                                      # 流水线多了一步：dws_user_hour（整表替换）
log[["step", "rows_in", "rows_out", "elapsed_sec"]].tail(2)

In [ ]:
# 对账：包里的版本 102,859 行，与 notebook 一致
assert len(user_hour) == 102_859
print("✅ 对账通过：dws.build_user_hour 与 notebook 版本逐行一致，102,859 行")

In [ ]:
# 需求：西塔、太原街、中街，一天里各在几点人最多？
# 有了 dws_user_hour，这就是一次分组计数。西塔按热点 500 米内算，太原街、中街
# 按 D3 选定的 v2 商圈边界算；一个位置同时满足时，先算西塔。
#
# 口径提醒：样本是"傍晚到访西塔的人"。太原街、中街的人数，只是这**群人**去那里
# 的人数，不是这两个商圈的总客流；西塔白天的人数也被低估了，因为白天来过、晚
# 上没来的人根本不在样本里。所以这张图只能比**形状**（几点高、几点低），不能比**高度**。
uh["zone"] = np.select([uh["dist_xita_m"] <= 500, uh["aoi"] == "太原街商圈", uh["aoi"] == "中街商圈"],
                       ["西塔", "太原街商圈", "中街商圈"], default="其他")
tab = (uh.groupby(["hour", "zone"]).size().unstack(fill_value=0)
       .reindex(index=range(24), fill_value=0)[["西塔", "太原街商圈", "中街商圈"]])
tab.T

In [ ]:
# ── 出图 D9_01：归一化热力矩阵 + 峰值条形图 ──────────────────────────────
# 为什么**不用三条折线叠在一张图上**：三个商圈的人数差了近 7 倍（西塔 2,037，中街 297），
# 叠在一起的话中街会被压成贴着横轴的一条直线，完全看不出它的节律。
# 解决办法是把问题拆成两个：
#   左边热力矩阵 —— 每行**除以自己的峰值**，只比"形状"（什么时候忙），行与行可比；
#   右边条形图   —— 补上被归一化抹掉的"高度"（峰值是多少人）。
# 这是处理"量级差很大的多条时间序列"的通用套路：形状和量级分开画。
# 规范落在哪几行：
#   · 按峰值时刻排序（ph.sort_values()）—— 排序本身就讲出了"高峰依次后移"这个结论
#   · 方框标出每行的峰值小时，并把时刻写在格子里 —— 不让读者去数列
#   · 右边条形用 AREA_COLOR 固定映射 —— 同一个区域在所有图里同色
#   · 副标题明写"不是商圈总客流，峰值人数不能直接比大小" ——
#     **这是本图最重要的一行**：样本是"西塔访客"，太原街和中街的数字只代表这群人
# 标题里的数字：ph（每个商圈的峰值小时），用 join 拼成一句话。
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
peak, ph = tab.max(), tab.idxmax()
order = ph.sort_values().index                                         # 按峰值时刻排序
norm_ = (tab / peak)[order].T                                          # 每个商圈除以自己的峰值：只比形状，不比高度
BLUES = LinearSegmentedColormap.from_list("blues", ["#EAF2F9", MAIN_MID, MAIN, MAIN_DARK])    # 单色渐变，不用彩虹色阶
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 5.6), layout="constrained", width_ratios=[3.2, 1])
a1.imshow(norm_.to_numpy(), aspect="auto", cmap=BLUES, vmin=0, vmax=1)
for i, kk in enumerate(order):
    a1.add_patch(Rectangle((ph[kk] - 0.5, i - 0.5), 1, 1, fill=False, ec=INK, lw=1.6))
    a1.text(ph[kk], i, f"{ph[kk]}点", ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")
a1.set_yticks(range(3), [kk.replace("商圈", "") for kk in order], fontsize=10.5, color=INK)
a1.set_xticks(range(0, 24, 3), [f"{x} 点" for x in range(0, 24, 3)]); a1.grid(False)
a1.set_title("每小时在场人数 ÷ 该商圈全天峰值（颜色越深越接近峰值）", loc="left", fontsize=10.5, color=INK, fontweight="bold")
bars = a2.barh(range(3), peak[order], color=[AREA_COLOR[kk] for kk in order], height=0.56)    # 区域按固定颜色
a2.bar_label(bars, labels=[f"{v:,.0f} 人" for v in peak[order]], padding=3, fontsize=9.5, color=INK)
a2.set_ylim(2.5, -0.5); a2.set_yticks([]); a2.set_xticks([]); a2.grid(False); a2.set_xlim(0, peak.max() * 1.55)
a2.set_title("峰值人数", loc="left", fontsize=10.5, color=INK, fontweight="bold")
title = "、".join(f"{kk.replace('商圈', '')} {ph[kk]} 点" for kk in order) + " 达到各自的峰值，西塔是夜间目的地"
viz.finish(fig, title,
           "样本用户每小时在三个商圈的人数（每人每小时一个位置，西塔 500 米内优先），按峰值时刻排序；方框 = 峰值小时；\n这是“西塔访客这群人”在三个商圈之间的时间分配，不是商圈总客流，峰值人数不能直接比大小；口径：dws_user_hour",
           SOURCE, "D9_01_三商圈的一天.png")

In [ ]:
# 这一格在做什么：把一张图写成一份能直接发出去的备忘录。
#
# 注意备忘录的结构，这就是企业里的标准写法：
#   标题 = 结论（不是"三商圈分析"）→ 写给谁 + 依据 → 三条发现 → 口径 → 局限 → 建议
# 两个容易被学生跳过、但最能体现专业度的部分：
#   · **口径**：每个数字是怎么算的，别人才能复核；
#   · **局限**：这份数据回答不了什么。不写局限的报告，一被追问就垮。
#
# 还有一点：所有数字都用 f-string 从变量里取（findings 那几行），不是手敲的。
# 数据一变，备忘录跟着变——和图表标题是同一个原则。
x, ty, zj = tab["西塔"], tab["太原街商圈"], tab["中街商圈"]
hx, hty, hzj = int(x.idxmax()), int(ty.idxmax()), int(zj.idxmax())
title = f"三个商圈的高峰依次错开：中街 {hzj} 点、太原街 {hty} 点、西塔 {hx} 点"
findings = [
    f"中街 {hzj} 点人最多（{zj[hzj]:,} 人），{hzj + 3} 点已降到 {zj[hzj + 3]:,} 人，是一个下午型商圈。",
    f"太原街全天在 {ty.min():,}–{ty.max():,} 人之间，最高只是最低的 {ty.max() / ty.min():.1f} 倍，三个商圈里最平稳。",
    f"西塔 {hx} 点达到高峰 {x[hx]:,} 人，23 点仍有 {x[23]:,} 人：数据截止时还没有散场。",
]
memo = f"""# {title}
**写给**：和平区商务局　　**依据**：2018 年 10 月 3 日，西塔傍晚访客样本（中国移动用户）

## 三条发现
1. {findings[0]}（图 D9_01）
2. {findings[1]}（图 D9_01）
3. {findings[2]}（图 D9_01）

## 口径
- 在场人数：每人每小时一个位置（覆盖该小时 30 分钟以上的停留优先，否则取信令中位位置），落在区域内即计 1 人；表 `dws_user_hour`。
- 西塔 = 热点 500 米内；太原街、中街 = v2 商圈边界。

## 局限
1. 样本是傍晚到访西塔的人：太原街、中街的数字只代表这群人，不是商圈总客流，三条线只能比形状。
2. 只有一天，而且是国庆假期：工作日、周末是不是同样的节律，这份数据回答不了。
3. 只有一家运营商的用户，位置是基站级的：城区误差几百米，商圈边界附近的人可能算错区域。

## 建议
1. 中街的活动放在 14–17 点，西塔的活动放在 18–21 点，同一群人的时间不冲突（对应发现 1、3）。由区商务局牵头排一张三商圈活动时间表。
2. 西塔商户按 18–23 点排班加人，数据截止时仍有上千人在场（对应发现 3）。
"""
config.REPORT_DIR.mkdir(exist_ok=True)
(config.REPORT_DIR / "备忘录示范_三商圈节律.md").write_text(memo, encoding="utf-8")
print(memo)

In [ ]:
# 这一格在做什么：把每人每小时的位置分进六边形格子，看少于 5 人的格子有多少。
#
# 为什么要先量一量再抑制：抑制是有代价的（信息变少了），得先知道代价多大。
# 结论是"被抑制的格子很多，但它们涉及的人次很少"——说明这个代价可以接受。
# 没有这一步，交付时客户问"你藏了多少数据"就答不上来。
#
# 下面那段 qf / rf / 取整纠偏是"平面坐标 → 六边形编号"的标准算法：
# 先算出小数坐标，四舍五入后用 q + r + s = 0 这个约束纠正误差最大的那一维。
# 细节不用让学生推导，知道"六边形网格有成熟算法、不要自己造"就够了。
S_KM = 0.55                                        # 六边形边长（公里）
EXT = (-12.5, 14.0, -12.0, 11.5)                   # 中心城区视窗：以西塔为原点，单位公里，西→东、南→北
kx, ky = 111.320 * np.cos(np.radians(hot["lat"])), 110.574       # 1 度经度、纬度各约多少公里
x = (uh["lng"] - hot["lng"]) * kx
y = (uh["lat"] - hot["lat"]) * ky
inside = x.between(EXT[0], EXT[1]) & y.between(EXT[2], EXT[3])
# 平面坐标 → 六边形编号：先算小数坐标，再取离它最近的那个格子
q_f, r_f = (np.sqrt(3) / 3 * x - y / 3) / S_KM, (2 / 3 * y) / S_KM
s_f = -q_f - r_f
q, r, s_ = q_f.round(), r_f.round(), s_f.round()
dq, dr, ds = (q - q_f).abs(), (r - r_f).abs(), (s_ - s_f).abs()
fix_q = (dq > dr) & (dq > ds)
fix_r = ~fix_q & (dr > ds)
q[fix_q] = -r[fix_q] - s_[fix_q]
r[fix_r] = -q[fix_r] - s_[fix_r]
cells = pd.DataFrame({"q": q, "r": r, "hour": uh["hour"]})[inside].astype(int)
cnt = cells.groupby(["q", "r", "hour"]).size()
K = 5
small = cnt < K
print(f"视窗内的人次占全部的 {inside.mean():.1%}")
print(f"非空的“格子 × 小时”共 {len(cnt):,} 个，不足 {K} 人的 {small.sum():,} 个（{small.mean():.1%}），"
      f"它们只涉及 {cnt[small].sum() / cnt.sum():.1%} 的人次")

In [ ]:
# 对账
assert round(small.mean(), 3) == 0.588 and round(cnt[small].sum() / cnt.sum(), 3) == 0.110
print("✅ 对账通过：58.8% 的格子被抑制，只涉及 11.0% 的人次")

In [ ]:
# ── 出图 D9_02：两条折线 ───────────────────────────────────────────────
# 为什么这两条线可以画在同一个坐标系里（而 D1_02、D4_05 要分上下两个面板）：
# 这两条都是**百分比**，单位和量级都一样，共用一个 0–100% 的纵轴没有任何误导。
# 判断标准就一条：**单位和量级相同 → 可以共轴；不同 → 分面板，绝不用双轴。**
# 规范落在哪几行：
#   · set_ylim(0, 100) 固定纵轴 —— 百分比的轴就该是 0–100%，自动缩放会夸大波动
#   · 线尾 annotate 直接标名字 —— 两条线不用图例
#   · 刻度写成 "0 / 25% / 50%..." —— 百分比就写百分号
# 标题里的数字：small.mean()（被抑制的格子占比）、cnt[small].sum() / cnt.sum()（涉及的人次占比）。
by_h = pd.DataFrame({"被抑制的格子": small.groupby(level="hour").mean(),
                     "被抑制的人次": cnt[small].groupby(level="hour").sum() / cnt.groupby(level="hour").sum()}) * 100
fig, ax = plt.subplots(figsize=(10, 5.4))
ax.plot(by_h.index, by_h["被抑制的格子"], color=GRAY, lw=2.4)
ax.plot(by_h.index, by_h["被抑制的人次"], color=MAIN, lw=2.4)
for col, color in [("被抑制的格子", "#6B7280"), ("被抑制的人次", MAIN)]:
    ax.annotate(f"{col} 占比", (23, by_h[col].iloc[-1]), xytext=(8, 0), textcoords="offset points",
                va="center", fontsize=10.5, color=color, fontweight="bold", annotation_clip=False)
ax.set_xlim(0, 23)
ax.set_ylim(0, 100)
ax.set_xticks(range(0, 24, 3), [f"{h} 点" for h in range(0, 24, 3)])
ax.set_yticks(range(0, 101, 25), [f"{v}%" for v in range(0, 101, 25)])
title = f"{small.mean():.1%} 的格子因为不足 {K} 人被隐藏，但只涉及 {cnt[small].sum() / cnt.sum():.1%} 的人次"
viz.finish(fig, title,
           f"每小时中心城区六边形（边长 {S_KM} 公里）里，人数少于 {K} 的“格子 × 小时”占比，以及它们涉及的人次占比；口径：dws_user_hour",
           SOURCE, "D9_02_小样本抑制.png")

In [ ]:
# 先在 notebook 里写一个能用的版本（封装卡 9.2 会改它的接口再收进包）
def suppress(table, k=5):
    """交付前的小样本抑制：少于 k 的格子换成 "<k"，返回抑制后的表和被抑制的格子数。"""
    out = table.astype(object).where(table >= k, f"<{k}")
    return out, int((table < k).to_numpy().sum())
night_tab = (uh[uh["hour"] <= 5].groupby(["district", "hour"]).size()
             .unstack(fill_value=0).sort_values(0, ascending=False))
delivered, n_masked = suppress(night_tab)
print(f"凌晨各区县每小时可定位人数：{night_tab.size} 个格子里抑制了 {n_masked} 个")
delivered

In [ ]:
from signal_insight import privacy
k = settings["privacy"]["min_cell_users"]
long = night_tab.stack().rename("n_users").reset_index()                  # 宽表 → 长表：一行一个“区县 × 小时”
sup = privacy.suppress(long, "n_users", k)
shown = (sup.assign(n_users=sup["n_users"].map(lambda v: f"{v:.0f}" if pd.notna(v) else f"<{k}"))
         .pivot(index="district", columns="hour", values="n_users")
         .reindex(index=night_tab.index, columns=night_tab.columns))
print(f"privacy.suppress 抑制了 {int(sup['suppressed'].sum())} 个格子；notebook 版本 {n_masked} 个")
shown

In [ ]:
# 对账：抑制的格子数相同，展示出来的宽表逐格相同
assert int(sup["suppressed"].sum()) == n_masked
assert (shown.astype(str).to_numpy() == delivered.astype(str).to_numpy()).all()
print("✅ 对账通过：privacy.suppress 与 notebook 版本抑制了同样的格子")

In [ ]:
# 这一格在做什么：演示"扩样"——从样本人数推算实际人数。
#
# ⚠ 下面两个比例 **全是假设值**，不是真实数据。所以这一卡的输出是一张"假设 × 结果"表，
# 而不是一个数字。给客户看的时候必须连同假设一起给，否则就是编数据。
#
# 讲台提示：企业里客户一定会问"那实际有多少人"。正确的回答方式是
# "按 X% 的市场份额推算约 N 人，这个 X 需要运营商提供"，
# 而不是直接报一个看起来精确的数字。
n19 = int(tab.loc[19, "西塔"])
rows = []
for share in [0.5, 0.6, 0.7]:                       # 移动用户占比：演示用的假设值，实际要向运营商要
    for own in [0.90, 0.95]:                        # 持有并开机的比例：演示用的假设值
        rows.append({"移动用户占比（假设）": f"{share:.0%}", "持机开机比例（假设）": f"{own:.0%}",
                     "扩样后约（人）": int(round(n19 / share / own, -1))})
pd.DataFrame(rows)

In [ ]:
from signal_mysql import database as mysql
sql = """
SELECT hour, COUNT(*) AS n
FROM dws_user_hour
WHERE dist_xita_m <= 500
GROUP BY hour
ORDER BY hour
"""
by_sql = mysql.read_frame(sql, con).set_index("hour")["n"]
diff = int((by_sql - tab["西塔"]).abs().sum())
print(f"SQL 与 pandas 逐小时相差之和：{diff}")

In [ ]:
# 对账
assert diff == 0 and by_sql[19] == 2_037
print("✅ 对账通过：SQL 与 pandas 完全一致，19 点西塔 2,037 人")

In [ ]:
con.close()